# CS103: Design and Analysis of Algorithms - Lab Assignment: Backtracking Algorithms
**Department of Computer Science & Engineering, SVNIT Surat**

This notebook contains simple, clear, and complete solutions for all questions in `Backtracking_Lab.pdf`:
1. **Experiment 1 (Question 1.1)**: Constrained 8-Queens Problem
2. **Experiment 2 (Question 2.1)**: Constrained Sum of Subsets Problem
3. **Experiment 3 (Question 3.1)**: List-Constrained Graph Coloring

All implementations use simple Python standard data structures with explicit backtracking and pruning logic.

---
## Experiment 1: Constrained 8-Queens Problem (Question 1.1)

### Problem Requirements:
- Solve the 8-Queens problem on an $8 \times 8$ chessboard using backtracking.
- Support **forbidden cells** where queens cannot be placed.
- Support **pre-placed queens** whose positions must remain fixed.
- Require exactly 8 queens on the board.
- **Reject contradictory input** (e.g., pre-placed queens attacking each other or on forbidden cells).
- **Prune branches immediately** when a placed queen attacks any previously placed queen.
- Print all valid arrangements and count total valid solutions.
- Support symmetry reduction when permitted.

In [1]:
def solve_constrained_8_queens(forbidden=None, pre_placed=None, symmetry_reduction=False):
    """
    Backtracking solver for the Constrained 8-Queens Problem.
    
    Parameters:
    - forbidden: set of (row, col) tuples where queens cannot be placed.
    - pre_placed: dict {row: col} representing fixed queen positions.
    - symmetry_reduction: bool, if True and constraints allow, reduces symmetric branches.
    
    Returns:
    - solutions: list of board configurations (each is a list where index=row, value=col).
    """
    if forbidden is None:
        forbidden = set()
    if pre_placed is None:
        pre_placed = {}

    # Requirement 4: Reject contradictory inputs
    # Check if any pre-placed queen occupies a forbidden cell
    for r, c in pre_placed.items():
        if (r, c) in forbidden:
            raise ValueError(f"Contradictory input: Pre-placed queen at ({r}, {c}) is in a forbidden cell.")

    # Check if any two pre-placed queens attack each other
    placed_list = list(pre_placed.items())
    for i in range(len(placed_list)):
        for j in range(i + 1, len(placed_list)):
            r1, c1 = placed_list[i]
            r2, c2 = placed_list[j]
            if r1 == r2 or c1 == c2 or abs(r1 - r2) == abs(c1 - c2):
                raise ValueError(f"Contradictory input: Pre-placed queens ({r1}, {c1}) and ({r2}, {c2}) attack each other.")

    solutions = []
    board = [-1] * 8  # board[r] represents the column of the queen in row r

    def is_safe(r, c):
        # Forbidden position check
        if (r, c) in forbidden:
            return False
        # Requirement 5: Prune immediately when attacking a previously placed queen
        for prev_r in range(r):
            prev_c = board[prev_r]
            if prev_c == c or abs(prev_r - r) == abs(prev_c - c):
                return False
        return True

    def backtrack(r):
        # Base case: All 8 queens placed successfully
        if r == 8:
            solutions.append(list(board))
            return

        # If current row has a pre-placed queen, respect it
        if r in pre_placed:
            c = pre_placed[r]
            if is_safe(r, c):
                board[r] = c
                backtrack(r + 1)
                board[r] = -1
            return

        col_range = range(8)
        # Requirement 6: Symmetry reduction (reflect horizontally across middle columns if unconstrained)
        if r == 0 and symmetry_reduction and not pre_placed and not forbidden:
            col_range = range(4)  # Explore only first 4 columns of row 0

        for c in col_range:
            if is_safe(r, c):
                board[r] = c
                backtrack(r + 1)
                board[r] = -1

    backtrack(0)
    return solutions


def print_board(solution):
    """Prints an 8x8 chessboard arrangement cleanly."""
    for r in range(8):
        row_str = " ".join("Q" if solution[r] == c else "." for c in range(8))
        print(f"Row {r}: {row_str}")
    print()

In [2]:
print("=== Experiment 1: Constrained 8-Queens Demonstrations ===")

# Test 1: Standard 8-Queens (no constraints)
all_solutions = solve_constrained_8_queens()
print(f"1. Standard 8-Queens total valid arrangements: {len(all_solutions)}")

# Test 2: Pre-placed Queen at (0, 0) and forbidden cells
forbidden = {(1, 2), (2, 4), (3, 1)}
pre_placed = {0: 0}  # Queen at row 0, col 0
constrained_solutions = solve_constrained_8_queens(forbidden=forbidden, pre_placed=pre_placed)
print(f"2. Constrained arrangements (pre-placed (0,0), forbidden={(1,2),(2,4),(3,1)}): {len(constrained_solutions)}")
print("\nFirst valid arrangement:")
print_board(constrained_solutions[0])

# Test 3: Contradictory Input Handling
print("3. Testing Contradictory Input Detection:")
try:
    # Two pre-placed queens in attack position: (0, 0) and (1, 1) on same diagonal
    solve_constrained_8_queens(pre_placed={0: 0, 1: 1})
except ValueError as e:
    print("  Successfully rejected:", e)

try:
    # Pre-placed queen on a forbidden cell
    solve_constrained_8_queens(forbidden={(0, 3)}, pre_placed={0: 3})
except ValueError as e:
    print("  Successfully rejected:", e)

=== Experiment 1: Constrained 8-Queens Demonstrations ===
1. Standard 8-Queens total valid arrangements: 92
2. Constrained arrangements (pre-placed (0,0), forbidden=((1, 2), (2, 4), (3, 1))): 3

First valid arrangement:
Row 0: Q . . . . . . .
Row 1: . . . . Q . . .
Row 2: . . . . . . . Q
Row 3: . . . . . Q . .
Row 4: . . Q . . . . .
Row 5: . . . . . . Q .
Row 6: . Q . . . . . .
Row 7: . . . Q . . . .

3. Testing Contradictory Input Detection:
  Successfully rejected: Contradictory input: Pre-placed queens (0, 0) and (1, 1) attack each other.
  Successfully rejected: Contradictory input: Pre-placed queen at (0, 3) is in a forbidden cell.



---
## Experiment 2: Sum of Subsets Problem (Question 2.1)

### Problem Requirements:
- Given a set of $n$ positive integers $S$ and a target sum $T$, find all subsets summing to $T$.
- Enforce **cardinality constraints**: minimum size $k_{\min}$ and maximum size $k_{\max}$.
- Enforce **mandatory elements** (must be present in every valid subset).
- Enforce **excluded elements** (must not be present in any valid subset).
- Handle duplicate values correctly by **distinguishing element positions / indices**.
- **Reject invalid input** (overlapping mandatory/excluded elements, invalid bounds, etc.).
- Pruning rules:
  1. Prune branch when current sum exceeds $T$.
  2. Prune branch when maximum possible sum of remaining eligible elements cannot reach $T$.
  3. Prune branch using cardinality bounds ($> k_{\max}$ or insufficient remaining to reach $k_{\min}$).
- **Benchmark Test Case**:
  - $S = \{3, 5, 6, 7, 9, 11, 12\}, T = 21$
  - Minimum size $k_{\min} = 2$, Maximum size $k_{\max} = 4$
  - Mandatory: $3$, Excluded: $7$
  - Report every valid subset and explain why each rejected branch was pruned.

In [3]:
def solve_constrained_sum_of_subsets(S, T, k_min, k_max, mandatory=None, excluded=None):
    """
    Backtracking solver for the Constrained Sum of Subsets Problem.
    
    Parameters:
    - S: list of positive integers.
    - T: target sum.
    - k_min: minimum permitted subset size.
    - k_max: maximum permitted subset size.
    - mandatory: list of element values that must appear.
    - excluded: list of element values that cannot appear.
    
    Returns:
    - valid_subsets: list of valid subsets (each as a list of values).
    - pruning_logs: list of (current_subset, reason) explaining pruned branches.
    """
    if mandatory is None:
        mandatory = []
    if excluded is None:
        excluded = []

    mand_set = set(mandatory)
    excl_set = set(excluded)

    # Requirement 4: Reject invalid input
    if mand_set & excl_set:
        raise ValueError("Invalid input: Mandatory and excluded elements overlap.")
    if k_min > k_max or k_min < 0:
        raise ValueError("Invalid input: k_min cannot exceed k_max.")
    for m in mand_set:
        if m not in S:
            raise ValueError(f"Invalid input: Mandatory element {m} is not in input set S.")

    # Requirement 3: Distinguish duplicate values by tracking (index, value)
    items = [(i, val) for i, val in enumerate(S) if val not in excl_set]

    valid_subsets = []
    pruning_logs = []

    def backtrack(idx, current_indices, current_sum):
        current_vals = [S[i] for i in current_indices]

        # Check if target sum is met
        if current_sum == T:
            # Check cardinality constraint
            if k_min <= len(current_indices) <= k_max:
                # Check mandatory elements requirement
                if mand_set.issubset(set(current_vals)):
                    valid_subsets.append(list(current_vals))
            # Since all elements are positive, adding further elements will exceed T
            return

        if idx >= len(items):
            return

        # Pruning 1: Current sum exceeds T
        if current_sum > T:
            pruning_logs.append((current_vals, f"Current sum ({current_sum}) exceeds target ({T})"))
            return

        # Pruning 7a: Cardinality limit exceeded
        if len(current_indices) >= k_max:
            pruning_logs.append((current_vals, f"Subset size reached maximum permitted cardinality k_max={k_max}"))
            return

        # Pruning 7b: Cannot satisfy minimum cardinality k_min
        remaining_count = len(items) - idx
        if len(current_indices) + remaining_count < k_min:
            pruning_logs.append((current_vals, f"Remaining elements ({remaining_count}) cannot reach minimum cardinality k_min={k_min}"))
            return

        # Pruning 6: Maximum possible sum of remaining eligible elements cannot reach T
        remaining_sum = sum(val for _, val in items[idx:])
        if current_sum + remaining_sum < T:
            pruning_logs.append((current_vals, f"Max achievable sum ({current_sum + remaining_sum}) is less than target ({T})"))
            return

        item_idx, item_val = items[idx]

        # Branch 1: Include items[idx]
        # Pruning 5: Prune immediately if adding item_val exceeds T
        if current_sum + item_val <= T:
            backtrack(idx + 1, current_indices + [item_idx], current_sum + item_val)
        else:
            pruning_logs.append((current_vals + [item_val], f"Adding element {item_val} gives sum {current_sum + item_val} > {T}"))

        # Branch 2: Exclude items[idx]
        backtrack(idx + 1, current_indices, current_sum)

    backtrack(0, [], 0)
    return valid_subsets, pruning_logs

In [4]:
print("=== Experiment 2: Benchmark Test Execution ===")

S = [3, 5, 6, 7, 9, 11, 12]
T = 21
k_min = 2
k_max = 4
mandatory = [3]
excluded = [7]

subsets, logs = solve_constrained_sum_of_subsets(S, T, k_min, k_max, mandatory, excluded)

print(f"Given Set S = {S}")
print(f"Target T = {T}")
print(f"Cardinality range: [{k_min}, {k_max}]")
print(f"Mandatory elements: {mandatory}")
print(f"Excluded elements: {excluded}")
print("-" * 50)

print(f"Total Valid Subsets Found: {len(subsets)}")
for idx, s in enumerate(subsets, 1):
    print(f"  Valid Subset {idx}: {s} (Sum = {sum(s)}, Size = {len(s)})")

print("\n" + "-" * 50)
print(f"Pruned Branches Report (Total Pruned: {len(logs)}):")
print("-" * 50)
for branch, reason in logs:
    branch_str = str(branch) if branch else "[]"
    print(f"Branch {branch_str.ljust(22)} -> {reason}")

=== Experiment 2: Benchmark Test Execution ===
Given Set S = [3, 5, 6, 7, 9, 11, 12]
Target T = 21
Cardinality range: [2, 4]
Mandatory elements: [3]
Excluded elements: [7]
--------------------------------------------------
Total Valid Subsets Found: 1
  Valid Subset 1: [3, 6, 12] (Sum = 21, Size = 3)

--------------------------------------------------
Pruned Branches Report (Total Pruned: 29):
--------------------------------------------------
Branch [3, 5, 6, 9]           -> Adding element 9 gives sum 23 > 21
Branch [3, 5, 6, 11]          -> Adding element 11 gives sum 25 > 21
Branch [3, 5, 6, 12]          -> Adding element 12 gives sum 26 > 21
Branch [3, 5, 9, 11]          -> Adding element 11 gives sum 28 > 21
Branch [3, 5, 9, 12]          -> Adding element 12 gives sum 29 > 21
Branch [3, 5, 11, 12]         -> Adding element 12 gives sum 31 > 21
Branch [3, 5]                 -> Max achievable sum (20) is less than target (21)
Branch [3, 6, 9, 11]          -> Adding element 11 gives 

---
## Experiment 3: Graph Coloring Using Backtracking (Question 3.1)

### Problem Requirements:
- Given an undirected graph $G = (V, E)$, assign colors to vertices such that adjacent vertices have different colors.
- Support **maximum $m$ available colors** (colors labeled $1, 2, \dots, m$).
- Support **vertex-specific allowed color lists** (`allowed_colors[v]`).
- Support **pre-colored vertices** whose assignments are fixed.
- **Reject invalid inputs** (pre-colored conflicts between adjacent vertices or colors outside allowed lists).
- Ensure each pre-colored vertex retains its assigned color.
- Ensure each vertex receives a color strictly from its allowed list.
- **Prune branches as soon as a color violates an adjacency or list constraint**.
- Report the total number of **recursive calls** and **branches pruned**.

In [5]:
def solve_list_constrained_graph_coloring(adj, m, allowed_colors, pre_colored=None):
    """
    Backtracking solver for List-Constrained Graph Coloring.
    
    Parameters:
    - adj: dict {vertex: [neighbors]} representing an undirected graph.
    - m: maximum available colors (1 to m).
    - allowed_colors: dict {vertex: [allowed_color_integers]}.
    - pre_colored: dict {vertex: fixed_color}.
    
    Returns:
    - solutions: list of valid colorings (each is a dict {vertex: color}).
    - stats: dict containing 'recursive_calls' and 'pruned_branches'.
    """
    if pre_colored is None:
        pre_colored = {}

    vertices = list(adj.keys())
    n = len(vertices)

    # Requirement 4 & Input validation: Check pre-colored vertices
    for u, col in pre_colored.items():
        if col not in allowed_colors.get(u, []):
            raise ValueError(f"Contradictory input: Pre-colored vertex {u} has color {col} not in its allowed list.")
        if col > m or col < 1:
            raise ValueError(f"Contradictory input: Pre-colored vertex {u} has color {col} exceeding available colors {m}.")
        # Check pre-colored adjacency conflict
        for v in adj[u]:
            if v in pre_colored and pre_colored[v] == col:
                raise ValueError(f"Contradictory input: Adjacent pre-colored vertices {u} and {v} both have color {col}.")

    colors = {u: None for u in vertices}
    for u, col in pre_colored.items():
        colors[u] = col

    solutions = []
    stats = {"recursive_calls": 0, "pruned_branches": 0}

    def is_safe(u, col):
        # Requirement 3: Reject any assignment in which adjacent vertices share a color
        for v in adj[u]:
            if colors[v] == col:
                return False
        return True

    def backtrack(idx):
        stats["recursive_calls"] += 1

        # Base case: All vertices colored
        if idx == n:
            solutions.append(dict(colors))
            return

        u = vertices[idx]

        # If vertex is pre-colored, preserve its color and verify adjacency
        if u in pre_colored:
            if is_safe(u, pre_colored[u]):
                backtrack(idx + 1)
            else:
                stats["pruned_branches"] += 1
            return

        # Requirement 5: Try only colors from the vertex's allowed list up to m
        for col in allowed_colors.get(u, []):
            if col > m:
                continue
            if is_safe(u, col):
                colors[u] = col
                backtrack(idx + 1)
                colors[u] = None
            else:
                # Requirement 6: Prune branch as soon as color violates adjacency
                stats["pruned_branches"] += 1

    backtrack(0)
    return solutions, stats

In [6]:
print("=== Experiment 3: List-Constrained Graph Coloring Demonstrations ===")

# Undirected graph G with 5 vertices (V0 to V4):
# Edges: (0-1), (0-2), (1-2), (1-3), (2-4), (3-4)
adj = {
    0: [1, 2],
    1: [0, 2, 3],
    2: [0, 1, 4],
    3: [1, 4],
    4: [2, 3]
}

m = 3  # Colors: 1, 2, 3

# Vertex-specific allowed color lists
allowed_colors = {
    0: [1, 2],
    1: [1, 2, 3],
    2: [2, 3],
    3: [1, 3],
    4: [1, 2, 3]
}

# Pre-colored vertex: Vertex 0 is fixed to color 1
pre_colored = {0: 1}

colorings, stats = solve_list_constrained_graph_coloring(adj, m, allowed_colors, pre_colored)

print(f"Graph vertices: {list(adj.keys())}")
print(f"Max colors (m): {m}")
print(f"Allowed colors per vertex: {allowed_colors}")
print(f"Pre-colored vertices: {pre_colored}")
print("-" * 50)
print(f"Valid Colorings Found: {len(colorings)}")
for i, col_map in enumerate(colorings, 1):
    print(f"  Coloring {i}: {col_map}")

print("-" * 50)
print("Search Statistics:")
print(f"  Total recursive calls: {stats['recursive_calls']}")
print(f"  Total branches pruned: {stats['pruned_branches']}")

# Test Contradictory Input: Adjacent vertices pre-colored with same color
print("\nTesting Contradictory Input Detection:")
try:
    solve_list_constrained_graph_coloring(adj, m, allowed_colors, pre_colored={0: 1, 1: 1})
except ValueError as e:
    print("  Successfully rejected:", e)

=== Experiment 3: List-Constrained Graph Coloring Demonstrations ===
Graph vertices: [0, 1, 2, 3, 4]
Max colors (m): 3
Allowed colors per vertex: {0: [1, 2], 1: [1, 2, 3], 2: [2, 3], 3: [1, 3], 4: [1, 2, 3]}
Pre-colored vertices: {0: 1}
--------------------------------------------------
Valid Colorings Found: 4
  Coloring 1: {0: 1, 1: 2, 2: 3, 3: 1, 4: 2}
  Coloring 2: {0: 1, 1: 2, 2: 3, 3: 3, 4: 1}
  Coloring 3: {0: 1, 1: 2, 2: 3, 3: 3, 4: 2}
  Coloring 4: {0: 1, 1: 3, 2: 2, 3: 1, 4: 3}
--------------------------------------------------
Search Statistics:
  Total recursive calls: 13
  Total branches pruned: 9

Testing Contradictory Input Detection:
  Successfully rejected: Contradictory input: Adjacent pre-colored vertices 0 and 1 both have color 1.

